In [ ]:
import os
import nibabel as nib
import numpy as np

mask_dir = "/home/amenacer/Stage/data-4D/6/$FRAME"

output_dir = os.path.join(mask_dir, "merged_masks")
os.makedirs(output_dir, exist_ok=True)

mask1_files = sorted([f for f in os.listdir(mask_dir) if f.endswith("_ROIMask-1.nii")])

for mask1_file in mask1_files:
    base_name = mask1_file.replace("_ROIMask-1.nii", "")
    mask2_file = f"{base_name}_ROIMask-2.nii"

    path1 = os.path.join(mask_dir, mask1_file)
    path2 = os.path.join(mask_dir, mask2_file)

    # Vérifie que les deux fichiers existent
    if not (os.path.exists(path1) and os.path.exists(path2)):
        print(f"Manquant : {path1} ou {path2}")
        continue

    mask1_img = nib.load(path1)
    mask2_img = nib.load(path2)

    mask1_data = mask1_img.get_fdata()
    mask2_data = mask2_img.get_fdata()

    # On suppose ici que les masques sont binaires (0 ou 1)
    merged = np.zeros_like(mask1_data, dtype=np.uint8)
    merged[mask1_data > 0] = 1   # ROI 1 en 1
    merged[mask2_data > 0] = 2   # ROI 2 en 2
    # Si jamais il y a chevauchement (rare, mais au cas où), on peut mettre 3 :
    merged[(mask1_data > 0) & (mask2_data > 0)] = 3

    merged_img = nib.Nifti1Image(merged, mask1_img.affine, mask1_img.header)
    output_path = os.path.join(output_dir, f"{base_name}_merged.nii.gz")
    nib.save(merged_img, output_path)
    print(f"Fusion enregistré : {output_path}")


In [ ]:
import os
import nibabel as nib
import numpy as np

label_dir = "/home/amenacer/Stage/data-4D/hadia/seg/mask/"

for fname in os.listdir(label_dir):
    if fname.endswith('.nii') or fname.endswith('.nii.gz'):
        fpath = os.path.join(label_dir, fname)
        img = nib.load(fpath)
        data = img.get_fdata()
        # Remplace tous les voxels 3 par 2
        data[data == 3] = 2
        new_img = nib.Nifti1Image(data.astype(np.uint8), img.affine, img.header)
        nib.save(new_img, fpath)
        print(f"Corrigé : {fname}")


In [ ]:
import os
import nibabel as nib
import numpy as np

label_dir = "/home/amenacer/Stage/data-4D/hadia/seg/mask/"

for fname in os.listdir(label_dir):
    if fname.endswith('.nii') or fname.endswith('.nii.gz'):
        fpath = os.path.join(label_dir, fname)
        img = nib.load(fpath)
        data = img.get_fdata()
        out = np.zeros_like(data, dtype=np.uint8)

        # Le myocarde (ROI2) est 1
        out[data == 1] = 1
        # Le ventricule gauche (ROI1) est 2 (y compris chevauchement)
        out[(data == 2) | (data == 3)] = 2

        new_img = nib.Nifti1Image(out, img.affine, img.header)
        nib.save(new_img, fpath)
        print(f"Corrigé : {fname} (fond=0, myocarde=1, VG=2)")
